# Train TrOCR on IAM Handwriting Dataset

This notebook fine-tunes the Microsoft TrOCR model on the IAM Handwriting dataset using Hugging Face datasets (`Teklia/IAM-line`).

### ⚠️ IMPORTANT: Enable GPU ⚠️
Go to **Runtime** > **Change runtime type** > Select **T4 GPU**.

In [ ]:
import torch
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU NOT Detected! Please change runtime type to GPU.")
    raise RuntimeError("No GPU found. Training will be too slow.")

In [ ]:
# 1. Clone or Update the repository
import os

if os.path.exists('handwriting_recog'):
    %cd handwriting_recog
    !git pull origin main
else:
    !git clone https://github.com/Bhuvan-018/handwriting_recog
    %cd handwriting_recog

In [ ]:
# 2. Install dependencies
!pip install -r requirements.txt

In [ ]:
# 3. Run the training script
!python train_hf.py

In [ ]:
# 4. Push Model to Hugging Face Space (Recommended for Large Files)
# GitHub has a 100MB file limit (2GB repo limit), while Hugging Face Spaces support Git LFS for large models.
# We will push directly to your Hugging Face Space.

# Install Git LFS
!git lfs install

# Configure Git
!git config --global user.email "your_email@example.com" # Replace with your email
!git config --global user.name "Colab User" # Replace with your name

# Clone your Hugging Face Space
# You need a Write Token from https://huggingface.co/settings/tokens
HF_TOKEN = "YOUR_HF_WRITE_TOKEN" # @param {type:"string"}
SPACE_ID = "bhuvan-018/handwriting-recognition" # @param {type:"string"}

if HF_TOKEN == "YOUR_HF_WRITE_TOKEN":
    print("⚠️ Please enter your Hugging Face Write Token above!")
else:
    repo_url = f"https://{HF_TOKEN}@huggingface.co/spaces/{SPACE_ID}"
    !git clone {repo_url} space_repo
    
    # Copy model files to the Space repo
    !mkdir -p space_repo/models/trocr_finetuned_iam_hf
    !cp -r models/trocr_finetuned_iam_hf/* space_repo/models/trocr_finetuned_iam_hf/
    
    # Copy app files (ensure they are up to date)
    !cp app_gradio.py space_repo/app.py
    !cp requirements.txt space_repo/
    !cp -r utils space_repo/
    
    # Commit and Push
    %cd space_repo
    !git lfs track "*.bin"
    !git lfs track "*.safetensors"
    !git add .
    !git commit -m "Deploy fine-tuned model from Colab"
    !git push
    print("✅ Successfully deployed to Hugging Face Space!")